In [ ]:
# ===== INSTALL ALL REQUIRED PACKAGES =====
import subprocess
import sys

packages = [
    "langchain",
    "langchain-community",
    "langchain-core",
    "langchain-text-splitters",
    "langchain-openai",
    "pypdf",
    "sentence-transformers",
    "faiss-cpu",
    "openai",
    "requests",
    "tiktoken"
]

for pkg in packages:
    print(f"Installing {pkg}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("\n✅ All packages installed successfully!")

In [ ]:
!pip install langchain langchain-community langchain-core langchain-text-splitters langchain-openai pypdf sentence-transformers faiss-cpu openai requests tiktoken -q

In [ ]:
# ===== IMPORTS =====
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

print("✅ All imports successful!")

In [ ]:
# ===== CONFIGURATION =====

# 🔑 OpenRouter API Key - Get yours at https://openrouter.ai/keys
OPENROUTER_API_KEY = "your-openrouter-api-key-here"  # Replace with your key

# 📄 Path to your PDF file
PDF_PATH = r"C:\Users\Sandesh\Desktop\SteveAI\data\lab_results.pdf"  # Update this path

# 📁 Vector store save path
VECTOR_STORE_PATH = "faiss_vectorstore"

# 🤖 OpenRouter model to use (free & capable options below)
# Options:
#   "mistralai/mistral-7b-instruct"         - Fast, free tier available
#   "meta-llama/llama-3-8b-instruct"        - Good quality, free
#   "google/gemma-3-27b-it:free"            - Google's model, free
#   "microsoft/phi-3-mini-128k-instruct"    - Lightweight, fast
MODEL_NAME = "mistralai/mistral-7b-instruct"

print(f"✅ Config set — Model: {MODEL_NAME}")

In [ ]:
FILE_NAME = os.path.basename(PDF_PATH)  

In [ ]:
from langchain_community.document_loaders import UnstructuredFileLoader

def load_documents(file_path):
    """Load documents of various formats"""
    
    print(f"Loading file: {file_path}")

    loader = UnstructuredFileLoader(file_path)
    documents = loader.load()

    print(f"✅ Loaded {len(documents)} document(s)")
    return documents


# Usage
documents = load_documents(FILE_PATH)

In [ ]:
# ===== STEP 2: SPLIT INTO CHUNKS =====

def create_chunks(documents):
    """Split documents into smaller overlapping chunks for better retrieval"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    chunks = text_splitter.split_documents(documents)
    print(f"✅ Created {len(chunks)} chunks from {len(documents)} page(s)")
    return chunks

text_chunks = create_chunks(documents)

In [ ]:
# ===== STEP 3: SETUP EMBEDDINGS =====

def setup_embeddings():
    """Initialize HuggingFace embedding model (runs locally, no API needed)"""
    print("Loading embedding model (sentence-transformers/all-MiniLM-L6-v2)...")
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"},   # Change to "cuda" if GPU available
        encode_kwargs={"normalize_embeddings": True}
    )
    print("✅ Embedding model loaded!")
    return embeddings

embeddings = setup_embeddings()

In [ ]:
# ===== STEP 4: CREATE / LOAD VECTOR STORE =====

def create_vector_store(chunks, embeddings, path=VECTOR_STORE_PATH):
    """Create FAISS vector store or load existing one"""
    if os.path.exists(path):
        print(f"Found existing vector store at '{path}'. Loading...")
        vectorstore = FAISS.load_local(
            path, embeddings, allow_dangerous_deserialization=True
        )
        print("✅ Vector store loaded successfully!")
    else:
        print(f"Creating new vector store at '{path}'...")
        vectorstore = FAISS.from_documents(
            documents=chunks,
            embedding=embeddings
        )
        vectorstore.save_local(path)
        print("✅ Vector store created and saved!")
    return vectorstore

vectorstore = create_vector_store(text_chunks, embeddings)

In [ ]:
# ===== STEP 5: SETUP OPENROUTER LLM =====

def setup_openrouter_llm():
    """
    Initialize LLM via OpenRouter API.
    OpenRouter is compatible with the OpenAI SDK — we just change the base_url.
    """
    print(f"Connecting to OpenRouter with model: {MODEL_NAME}")

    llm = ChatOpenAI(
        model=MODEL_NAME,
        openai_api_key=OPENROUTER_API_KEY,
        openai_api_base="https://openrouter.ai/api/v1",
        temperature=0.3,
        max_tokens=1024,
        default_headers={
            "HTTP-Referer": "http://localhost",      # Optional: for OpenRouter analytics
            "X-Title": "RAG Medical Assistant"       # Optional: shown in OpenRouter dashboard
        }
    )
    print("✅ OpenRouter LLM ready!")
    return llm

llm = setup_openrouter_llm()

In [ ]:
# ===== STEP 6: SETUP RETRIEVER =====

def setup_retriever(vectorstore, k=5):
    """Create similarity-based retriever from vector store"""
    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k}
    )
    return retriever

retriever = setup_retriever(vectorstore, k=5)

# Quick test — verify retriever works
test_docs = retriever.invoke("Patient Medical Record")
print(f"✅ Retriever working — found {len(test_docs)} relevant chunk(s)")
print("\n--- Sample chunk preview ---")
print(test_docs[0].page_content[:300] if test_docs else "No chunks found")

In [ ]:
# ===== STEP 7: BUILD RAG CHAIN =====

PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""\
You are a helpful medical AI assistant.

Use ONLY the context provided below to answer the question.
If the answer cannot be found in the context, respond with: "I don't know based on the provided document."
Be concise and accurate.

Context:
{context}

Question:
{question}

Answer:
"""
)

def format_docs(docs):
    """Join retrieved document chunks into a single context string"""
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | PROMPT
    | llm
    | StrOutputParser()
)

print("✅ RAG chain built successfully!")

In [ ]:
# ===== STEP 8: QUERY FUNCTION =====

def ask_question(question: str) -> str:
    """
    Ask a question about the loaded PDF document.
    Returns the LLM answer using retrieved context.
    """
    print("\n" + "="*55)
    print(f"❓ Question: {question}")
    print("="*55)

    answer = rag_chain.invoke(question)

    print(f"💬 Answer:\n{answer}")
    print("="*55)
    return answer

In [ ]:
# ===== STEP 9: RUN QUESTIONS =====

questions = [
    "What is the main topic of this document?",
    "What are the key findings?",
    "All details about the patient.",
    "What are the allergies of the patient?",
    "What is the age of the patient?",
    "What are the date of joining and discharge date of the patient?"
]

for question in questions:
    ask_question(question)